# Coordinate Transformations

좌표 변환 정의 → Jacobian 자동 계산 → metric/tensor 변환.

$$g'_{\mu\nu} = \frac{\partial x^\alpha}{\partial x'^\mu}\frac{\partial x^\beta}{\partial x'^\nu}\, g_{\alpha\beta}, \qquad V'^\mu = \frac{\partial x'^\mu}{\partial x^\alpha}\, V^\alpha$$

In [ ]:
import sys; sys.path.insert(0, "/home/minjae/Minjae/IndexCalc")
import sympy as sp
from indexcalc import CoordinateTransform, Metric

## 1. Spherical → Cartesian: Jacobian & Metric 변환

In [ ]:
r, θ, φ = sp.symbols('r θ φ', positive=True)

T = CoordinateTransform(
    source=["r", "θ", "φ"],
    target=["x", "y", "z"],
    forward=[r*sp.sin(θ)*sp.cos(φ), r*sp.sin(θ)*sp.sin(φ), r*sp.cos(θ)],
    source_symbols=[r, θ, φ],
)

print("Jacobian J = ∂(x,y,z)/∂(r,θ,φ):")
sp.pprint(T.jacobian())

print("\nJ⁻¹:")
sp.pprint(sp.simplify(T.jacobian_inv()))

# Metric 변환
g_sph = sp.diag(1, r**2, r**2 * sp.sin(θ)**2)
g_cart = T.transform_metric(Metric(g_sph, ["r","θ","φ"]))

print(f"\ng_cart = {sp.simplify(g_cart._sympy_metric)}")
assert sp.simplify(g_cart._sympy_metric) == sp.eye(3)
print("✓ ds² = dr² + r²dΩ² → dx² + dy² + dz²")

## 2. Vector & Covector 변환

In [ ]:
# e_r → Cartesian
e_r = sp.Matrix([1, 0, 0])
e_r_cart = T.transform_components(e_r, rank=(1, 0))
print(f"e_r (cart) = {sp.simplify(e_r_cart).T}")

# Covector dr → Cartesian
dr = sp.Matrix([1, 0, 0])
dr_cart = T.transform_components(dr, rank=(0, 1))
print(f"dr  (cart) = {sp.simplify(dr_cart).T}")

# (1,1) tensor: identity → identity
I = sp.eye(3)
I_cart = T.transform_components(I, rank=(1, 1))
print(f"\nδ^μ_ν 변환 = {sp.simplify(I_cart)}")
assert sp.simplify(I_cart) == sp.eye(3)
print("✓ 단위 텐서 불변")

## 3. LaTeX 출력

In [ ]:
from IPython.display import display, Math
display(Math(T.latex()))

## 4. Numeric (JAX): Polar → Cartesian + Curvature 불변

In [ ]:
import jax; jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

def polar_to_cart(x):
    return jnp.array([x[0]*jnp.cos(x[1]), x[0]*jnp.sin(x[1])])

def cart_to_polar(x):
    return jnp.array([jnp.sqrt(x[0]**2+x[1]**2), jnp.arctan2(x[1],x[0])])

T_num = CoordinateTransform(["r","θ"], ["x","y"],
    forward=polar_to_cart, inverse=cart_to_polar)

def polar_metric(x):
    return jnp.diag(jnp.array([1.0, x[0]**2]))

g_polar = Metric(polar_metric, ["r","θ"])
g_cart = T_num.transform_metric(g_polar)

result = g_cart.at(jnp.array([1.0, 1.0]))
print(f"g_cart at (1,1) =\n{result.metric}")
print(f"R = {float(result.R):.2e}")
print("✓ 평탄 metric, R=0")